In [137]:
from urllib.parse import urlparse
from pathlib import Path
import re
import requests
import pandas as pd

resp = requests.get("https://mise-java.jdx.dev/jvm/ga/macosx/aarch64.json")
resp.raise_for_status()

data = resp.json()
df = pd.DataFrame(data)

def version_key(v: str, width: int = 4, include_build: bool = True) -> tuple[int, ...]:
    """
    Examples:
      "21.0.10"      -> (21, 0, 10, 0, 0)  # build=0
      "8.0.345"      -> (8, 0, 345, 0, 0)
      "11.0.30+7"    -> (11, 0, 30, 0, 7)
      "11.0.20.1+1"  -> (11, 0, 20, 1, 1)
      "11.0.22+7.1"  -> (11, 0, 22, 0, 7)  # "+7.1"이면 build는 첫 숫자만
    """
    s = "" if pd.isna(v) else str(v).strip()

    m = re.match(r"(?P<num>\d+(?:\.\d+)*)(?:\+(?P<build>\d+)(?:\.\d+)*)?", s)
    if not m:
        base = [0] * width
        return tuple(base + ([0] if include_build else []))

    num = m.group("num") or ""
    build = int(m.group("build")) if (include_build and m.group("build")) else 0

    parts = [int(x) for x in num.split(".") if x != ""]
    parts = (parts + [0] * width)[:width]

    return tuple(parts + ([build] if include_build else []))

# df["vendor"].drop_duplicates()
# df[(df["vendor"] == "zulu") & (df["file_type"] == "tar.gz")].assign(java_version_key=lambda d: d["java_version"].map(version_key)).sort_values(by="java_version_key")


# jdk install zulu-21
# jdk install zulu-17
# jdk install zulu-11
# jdk install zulu-8

def some_function(x: str):
    """
    zulu-21, zulu-17, zulu-11, zulu-8

    """

# get "zulu-21"
input_vendor = "zulu"
input_major = 11

df = df.loc[
    (df["vendor"] == input_vendor) &
    (df["image_type"] == "jdk") &
    (df["file_type"] == "tar.gz") &
    (df["features"].str.len() == 0)
]
df = df.assign(java_version_key=lambda d: d["java_version"].map(version_key))
#df["main_name"] = df["vendor"] + "-" + df["java_version_key"].str[0]
df = df.loc[df["java_version_key"].str[0] == input_major].sort_values(by=["java_version_key", "version"], ascending=[False, False])
df


,checksum,created_at,features,file_type,image_type,java_version,jvm_impl,url,vendor,version,java_version_key
3798,sha256:a6feb4c541c9f4f1d3b4bdcca2a25ca39b8d2d7...,2026-02-03T22:07:02.246803,[],tar.gz,jdk,11.0.30,hotspot,https://cdn.azul.com/zulu/bin/zulu11.86.21-ca-...,zulu,11.86.21.0,"(11, 0, 30, 0, 0)"
3820,sha256:383c6390f6ca2ad6d5309ec0239cb6bafb9fa40...,2026-02-03T22:07:02.246803,[],tar.gz,jdk,11.0.30,hotspot,https://cdn.azul.com/zulu/bin/zulu11.86.19-ca-...,zulu,11.86.19.0,"(11, 0, 30, 0, 0)"
3434,sha256:09ed1734c2d88fadcb75fdbec1ba5467d32e7fa...,2025-10-22T22:02:52.118933,[],tar.gz,jdk,11.0.29,hotspot,https://cdn.azul.com/zulu/bin/zulu11.84.17-ca-...,zulu,11.84.17.0,"(11, 0, 29, 0, 0)"
3132,sha256:5b104e96bb41dc38b1605d701e4482003acffbe...,2025-07-15T22:02:54.266547,[],tar.gz,jdk,11.0.28,hotspot,https://cdn.azul.com/zulu/bin/zulu11.82.19-ca-...,zulu,11.82.19.0,"(11, 0, 28, 0, 0)"
3011,sha256:f24b8c06b586efbde158200e764e89d7437f291...,2025-04-16T22:03:04.613565,[],tar.gz,jdk,11.0.27,hotspot,https://cdn.azul.com/zulu/bin/zulu11.80.21-ca-...,zulu,11.80.21.0,"(11, 0, 27, 0, 0)"
701,sha256:3708badcc0c79fc1791e74b62478188a1f43c4f...,2025-03-28T22:02:46.826962,[],tar.gz,jdk,11.0.26,hotspot,https://cdn.azul.com/zulu/bin/zulu11.78.15-ca-...,zulu,11.78.15.0,"(11, 0, 26, 0, 0)"
689,sha256:e62bdbbaa97d631933554e59a842f0e94b2355f...,2025-03-28T22:02:46.826962,[],tar.gz,jdk,11.0.25,hotspot,https://cdn.azul.com/zulu/bin/zulu11.76.21-ca-...,zulu,11.76.21.0,"(11, 0, 25, 0, 0)"
3,sha256:f8ac458076c10f13753b7342033aaa073b715ef...,2025-03-28T22:02:46.826962,[],tar.gz,jdk,11.0.24,hotspot,https://cdn.azul.com/zulu/bin/zulu11.74.15-ca-...,zulu,11.74.15.0,"(11, 0, 24, 0, 0)"
840,sha256:40fb1918385e03814b67b7608c908c7f945ccbe...,2025-03-28T22:02:46.826962,[],tar.gz,jdk,11.0.23,hotspot,https://cdn.azul.com/zulu/bin/zulu11.72.19-ca-...,zulu,11.72.19.0,"(11, 0, 23, 0, 0)"
251,sha256:3d640e17e3fd76365aae3009684dc8d2db88d44...,2025-03-28T22:02:46.826962,[],tar.gz,jdk,11.0.22,hotspot,https://cdn.azul.com/zulu/bin/zulu11.70.15-ca-...,zulu,11.70.15.0,"(11, 0, 22, 0, 0)"


In [129]:
df = df.head(1)
download_url = df.iloc[0]["url"]
print(download_url)

https://github.com/adoptium/temurin17-binaries/releases/download/jdk-17.0.18%2B8/OpenJDK17U-jdk_aarch64_mac_hotspot_17.0.18_8.tar.gz


In [130]:
# 다운로드

dest_dir = Path("/Users/xuny/Downloads")
dest_dir.mkdir(parents=True, exist_ok=True)

filename = Path(urlparse(download_url).path).name  # URL의 마지막 경로를 파일명으로
dest_path = dest_dir / filename

with requests.get(download_url, stream=True, timeout=60) as r:
    r.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1MB
            if chunk:  # keep-alive chunk 방지
                f.write(chunk)

dest_path.as_posix()

'/Users/xuny/Downloads/OpenJDK17U-jdk_aarch64_mac_hotspot_17.0.18_8.tar.gz'

In [132]:
# 다운로드 받은 파일의 checksum 체크
import hashlib
from pathlib import Path

def sha256_file(path: str | Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

actual = sha256_file(dest_path)   # dest_path = 다운로드해서 저장한 파일 경로
print(actual)

###

expected = df.iloc[0]["checksum"]          # 1행짜리 df에서 checksum 가져온다고 가정
expected_hex = str(expected).removeprefix("sha256:").lower()

actual_hex = sha256_file(dest_path).lower()

ok = (actual_hex == expected_hex)
ok, expected_hex, actual_hex

d81de06d938384fe76c4aa3c13395933aa11e2d19b0428743f810db06b05e312


(True,
 'd81de06d938384fe76c4aa3c13395933aa11e2d19b0428743f810db06b05e312',
 'd81de06d938384fe76c4aa3c13395933aa11e2d19b0428743f810db06b05e312')

In [ ]:
# 다운로드 받은 파일을 압축해제
# /Users/xuny/Downloads/zulu17.64.17-ca-jdk17.0.18-macosx_aarch64.tar.gz
